# ai.wasm を Python から使うサンプル

## 概要

`ai.wasm` は Emscripten でコンパイルされたオセロAI（Egaroucid）です。  
WASM 本体と JS グルーコード（`ai.js`）がセットで動作するため、  
Python から直接 WASM を読む代わりに **Node.js をサブプロセスとして呼び出す**方式を採用しています。

## 前提
- Node.js がインストールされていること
- `js/ai.wasm` と `js/ai.js` がこのノートブックの隣の `js/` フォルダにあること

## 盤面の表現（Python側）
```
board[y][x]  ← 8×8 の二次元リスト
 1  = 黒石
-1  = 白石
 0  = 空マス

座標: (x=0, y=0) が左上 (a1)、(x=7, y=7) が右下 (h8)
```

## WASM の主なエクスポート関数
| 関数 | 引数 | 返り値 |
|------|------|--------|
| `_init_ai(ptr)` | 空ポインタ | 0=成功 |
| `_ai_js(boardPtr, level, player)` | 盤面・深さ・手番 | エンコード済み整数 |
| `_calc_value(boardPtr, level, player)` | 同上 | 評価値のみ |

---
## 1. 準備：Node.js の確認

In [37]:
import subprocess
import json
import os
import tempfile
from pathlib import Path

# このノートブックが置かれているディレクトリを基準にパスを設定
NOTEBOOK_DIR = Path(os.getcwd())
AI_JS_PATH   = NOTEBOOK_DIR / 'js' / 'ai.js'

# Node.js バージョン確認
result = subprocess.run(['node', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'Node.js: {result.stdout.strip()}')
else:
    raise RuntimeError('Node.js が見つかりません。インストールしてください。')

# ai.js の存在確認
if AI_JS_PATH.exists():
    print(f'ai.js: {AI_JS_PATH}')
else:
    raise FileNotFoundError(f'ai.js が見つかりません: {AI_JS_PATH}')

Node.js: v20.11.0
ai.js: c:\Users\morup\Documents\オセロ\othello-yj.github.io\js\ai.js


---
## 2. Node.js ブリッジスクリプトの定義

Python から `subprocess` で呼ぶ Node.js スクリプトを一時ファイルに書き出します。  
このスクリプトは:
1. ai.js を `require()` して AI を初期化
2. コマンドライン引数で受け取った盤面を評価
3. 結果を JSON で stdout に出力

In [38]:
NODE_BRIDGE_SCRIPT = r"""
'use strict';
const path = require('path');
const vm   = require('vm');
const fs   = require('fs');
const util = require('util');

const [aiJsPath, cmdJson] = process.argv.slice(2);
const cmd = JSON.parse(cmdJson);

function output(obj) {
    process.stdout.write(JSON.stringify(obj) + '\n');
    process.exit(0);
}

// ===== 盤面エンコード =====
function encodeBoard(M, board) {
    const arr = new Int32Array(64);
    for (let y = 0; y < 8; y++)
        for (let x = 0; x < 8; x++) {
            const v = board[y][x];
            arr[y * 8 + x] = v === 1 ? 0 : v === -1 ? 1 : -1;
        }
    const ptr = M._malloc(64 * 4);
    M.HEAP32.set(arr, ptr >> 2);
    return ptr;
}

// ===== _ai_js 戻り値デコード =====
function decodeAiResult(val) {
    const my        = Math.floor(val / 8000);
    const mx        = Math.floor((val - my * 8000) / 1000);
    const score_raw = val - my * 8000 - mx * 1000 - 100;
    return { mx, my, score_raw };
}

// ===== 合法手チェック =====
function hasValidMove(board, player) {
    const opp = -player;
    const DIRS = [[-1,-1],[-1,0],[-1,1],[0,-1],[0,1],[1,-1],[1,0],[1,1]];
    for (let y = 0; y < 8; y++)
        for (let x = 0; x < 8; x++) {
            if (board[y][x] !== 0) continue;
            for (const [dx, dy] of DIRS) {
                let nx = x+dx, ny = y+dy, found = false;
                while (nx>=0&&nx<8&&ny>=0&&ny<8&&board[ny][nx]===opp) { nx+=dx; ny+=dy; found=true; }
                if (found && nx>=0&&nx<8&&ny>=0&&ny<8&&board[ny][nx]===player) return true;
            }
        }
    return false;
}

// ===== 着手適用 =====
function applyMove(board, x, y, player) {
    const opp = -player;
    const DIRS = [[-1,-1],[-1,0],[-1,1],[0,-1],[0,1],[1,-1],[1,0],[1,1]];
    const b = board.map(r => [...r]);
    b[y][x] = player;
    for (const [dx, dy] of DIRS) {
        const line = [];
        let nx = x+dx, ny = y+dy;
        while (nx>=0&&nx<8&&ny>=0&&ny<8&&b[ny][nx]===opp) { line.push([nx,ny]); nx+=dx; ny+=dy; }
        if (line.length && nx>=0&&nx<8&&ny>=0&&ny<8&&b[ny][nx]===player)
            for (const [fx,fy] of line) b[fy][fx] = player;
    }
    return b;
}

// ===== コマンドディスパッチ =====
function dispatch(M, cmd) {
    const level      = cmd.level || 8;
    const wasmPlayer = cmd.player === 1 ? 0 : 1;

    if (cmd.type === 'best_move') {
        // 1手だけ最善手と評価値を返す
        const ptr = encodeBoard(M, cmd.board);
        const val = M._ai_js(ptr, level, wasmPlayer);
        M._free(ptr);
        const { mx, my, score_raw } = decodeAiResult(val);
        const score = wasmPlayer === 0 ? score_raw : -score_raw;
        return { mx, my, score, col: 'abcdefgh'[mx], row: my + 1 };

    } else if (cmd.type === 'calc_value') {
        // 評価値のみを返す（手は不要な場合）
        const ptr = encodeBoard(M, cmd.board);
        const val = M._ai_js(ptr, level, wasmPlayer);
        M._free(ptr);
        const { score_raw } = decodeAiResult(val);
        const score = wasmPlayer === 0 ? score_raw : -score_raw;
        return { score };

    } else if (cmd.type === 'solve_endgame') {
        // 終盤全読み：1回の呼び出しで終局まで打ち切る
        // WASM を使い回すことでプロセス起動・初期化コストを 1回に抑える
        let b  = cmd.board.map(r => [...r]);
        let cp = cmd.player;
        const moves = [];

        for (;;) {
            if (!hasValidMove(b, cp)) {
                if (!hasValidMove(b, -cp)) break;
                cp = -cp;
                continue;
            }
            const wp  = cp === 1 ? 0 : 1;
            const ptr = encodeBoard(M, b);
            const val = M._ai_js(ptr, level, wp);
            M._free(ptr);
            const { mx, my } = decodeAiResult(val);
            moves.push({ mx, my, col: 'abcdefgh'[mx], row: my + 1, player: cp });
            b  = applyMove(b, mx, my, cp);
            cp = -cp;
        }

        let black = 0, white = 0, empty = 0;
        for (const row of b) for (const v of row) {
            if (v === 1) black++; else if (v === -1) white++; else empty++;
        }
        if (black > white) black += empty;
        else if (white > black) white += empty;

        return { moves, black, white, score: black - white, final_board: b };

    } else {
        return { error: 'unknown command type: ' + cmd.type };
    }
}

// ===== AI 初期化 =====
function onReady() {
    const M = this;
    try {
        const initPtr = M._malloc(4);
        const initResult = M._init_ai(initPtr);
        M._free(initPtr);
        if (initResult !== 0) { output({ error: '_init_ai failed: ' + initResult }); return; }
        output(dispatch(M, cmd));
    } catch(e) {
        output({ error: String(e) });
    }
}

// ===== vm サンドボックスで ai.js を実行 =====
const aiJsSource = fs.readFileSync(path.resolve(aiJsPath), 'utf8');
const aiDir      = path.dirname(path.resolve(aiJsPath));

const sandbox = {
    process, Buffer,
    setTimeout, clearTimeout, setInterval, clearInterval, setImmediate, clearImmediate,
    console: { log: ()=>{}, warn: ()=>{}, error: ()=>{}, info: ()=>{} },
    require: (id) => {
        if (id.startsWith('.')) return require(path.resolve(aiDir, id));
        return require(id);
    },
    __filename: path.resolve(aiJsPath),
    __dirname:  aiDir,
    WebAssembly,
    TextDecoder: typeof TextDecoder !== 'undefined' ? TextDecoder : util.TextDecoder,
    TextEncoder: typeof TextEncoder !== 'undefined' ? TextEncoder : util.TextEncoder,
    performance: typeof performance !== 'undefined' ? performance : require('perf_hooks').performance,
    Module: { print: ()=>{}, printErr: ()=>{}, onRuntimeInitialized: onReady },
};

vm.createContext(sandbox);
vm.runInContext(aiJsSource, sandbox);
"""

import tempfile
from pathlib import Path

BRIDGE_FILE = Path(tempfile.mktemp(suffix='.js'))
BRIDGE_FILE.write_text(NODE_BRIDGE_SCRIPT, encoding='utf-8')
print(f'ブリッジスクリプト: {BRIDGE_FILE}')

ブリッジスクリプト: C:\Users\morup\AppData\Local\Temp\tmpbvj9u8o0.js


---
## 3. Python ヘルパー関数

In [39]:
def call_ai(cmd: dict, timeout: int = 30) -> dict:
    """Node.js ブリッジを呼び出し、結果を dict で返す。"""
    result = subprocess.run(
        ['node', str(BRIDGE_FILE), str(AI_JS_PATH), json.dumps(cmd)],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    # stdout・stderr 両方をデバッグ用に確認できるようにする
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(
            f'Node.js 失敗\n'
            f'  returncode : {result.returncode}\n'
            f'  stdout     : {repr(result.stdout[:300])}\n'
            f'  stderr     : {repr(result.stderr[:500])}'
        )
    return json.loads(result.stdout)


def best_move(board: list[list[int]], player: int, level: int = 8) -> dict:
    """
    Egaroucid に最善手を問い合わせる。

    Parameters
    ----------
    board  : 8×8 の二次元リスト (1=黒, -1=白, 0=空)
    player : 手番 (1=黒, -1=白)
    level  : 探索深さ (1〜21、デフォルト8)

    Returns
    -------
    dict: { mx: int, my: int, score: int, col: str, row: int }
          mx/my は 0-indexed 座標、col/row は棋譜表記 (例: 'd', 3)
    """
    return call_ai({'type': 'best_move', 'board': board, 'player': player, 'level': level})


def calc_value(board: list[list[int]], player: int, level: int = 8) -> int:
    """
    Egaroucid に局面の評価値（黒視点の予測石差）を問い合わせる。

    Returns
    -------
    int: 正=黒有利, 負=白有利
    """
    result = call_ai({'type': 'calc_value', 'board': board, 'player': player, 'level': level})
    return result['score']


# ===== 盤面ユーティリティ =====

def initial_board() -> list[list[int]]:
    """オセロの初期盤面を返す（中央4マスに初期配置）。"""
    b = [[0] * 8 for _ in range(8)]
    b[3][3] = -1  # d4: 白
    b[3][4] =  1  # e4: 黒
    b[4][3] =  1  # d5: 黒
    b[4][4] = -1  # e5: 白
    return b


def print_board(board: list[list[int]], last_move: tuple[int,int] | None = None) -> None:
    """盤面をテキストで表示する。"""
    print('  a b c d e f g h')
    for y, row in enumerate(board):
        line = f'{y+1} '
        for x, v in enumerate(row):
            if last_move and (x, y) == last_move:
                line += ('● ' if v == 1 else '○ ')
            elif v == 1:
                line += '● '
            elif v == -1:
                line += '○ '
            else:
                line += '. '
        print(line)
    black = sum(v == 1  for row in board for v in row)
    white = sum(v == -1 for row in board for v in row)
    print(f'   黒: {black}  白: {white}')


def get_flips(board, x, y, player):
    """(x,y) に player が打ったときにひっくり返る座標のリストを返す。"""
    opponent = -player
    flips = []
    for dx, dy in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]:
        line = []
        nx, ny = x + dx, y + dy
        while 0 <= nx < 8 and 0 <= ny < 8 and board[ny][nx] == opponent:
            line.append((nx, ny))
            nx += dx; ny += dy
        if line and 0 <= nx < 8 and 0 <= ny < 8 and board[ny][nx] == player:
            flips.extend(line)
    return flips


def apply_move(board, x, y, player):
    """(x,y) に player が打った後の新しい盤面を返す（元の盤面は変更しない）。"""
    import copy
    b = copy.deepcopy(board)
    b[y][x] = player
    for fx, fy in get_flips(board, x, y, player):
        b[fy][fx] = player
    return b


def valid_moves(board, player):
    """player が打てるすべての (x, y) を返す。"""
    return [(x, y)
            for y in range(8)
            for x in range(8)
            if board[y][x] == 0 and get_flips(board, x, y, player)]


print('ヘルパー関数定義完了')

ヘルパー関数定義完了


---
## 4. 動作確認：初期局面の最善手を求める

In [40]:
board = initial_board()
print('=== 初期局面 ===')
print_board(board)

print('\n黒番の最善手を計算中 (level=8)...')
result = best_move(board, player=1, level=8)
print(f'最善手: {result["col"]}{result["row"]}  (x={result["mx"]}, y={result["my"]})')
print(f'評価値: {result["score"]:+d}  (黒視点の予測石差)')

=== 初期局面 ===
  a b c d e f g h
1 . . . . . . . . 
2 . . . . . . . . 
3 . . . . . . . . 
4 . . . ○ ● . . . 
5 . . . ● ○ . . . 
6 . . . . . . . . 
7 . . . . . . . . 
8 . . . . . . . . 
   黒: 2  白: 2

黒番の最善手を計算中 (level=8)...
最善手: d3  (x=3, y=2)
評価値: +0  (黒視点の予測石差)


---
## 5. 評価値のみを取得する

In [41]:
board = initial_board()

# 黒の合法手それぞれを打った後の局面を評価
moves = valid_moves(board, player=1)
print('黒の合法手と評価値:')
print(f'{"手":>5}  {"評価値":>6}')
print('-' * 14)

for x, y in moves:
    # 一手打った後の局面で白番として評価（次は白の手番なので）
    next_board = apply_move(board, x, y, player=1)
    score = calc_value(next_board, player=-1, level=8)
    col = 'abcdefgh'[x]
    print(f'{col}{y+1:>4}  {score:>+6d}')

黒の合法手と評価値:
    手     評価値
--------------
d   3      +0
c   4      +0
f   5      +0
e   6      +0


---
## 6. AI 同士を対局させる

---
## 6. 終盤全読み（15手）

空きマスが少ない終盤局面では、Egaroucid に高いレベルを渡すことで完全読み（全読み）が可能です。  
両者最善手を交互に打ち切り、最終的な実石差を返します。

| レベル目安 | 空きマス数 |
|-----------|-----------|
| 10 | ≦20 |
| 15 | ≦17 |
| 21（全読み） | ≦15 |

In [44]:
import time

def solve_endgame(board: list[list[int]], player: int, level: int = 21) -> dict:
    """
    終盤局面を全読みして最終結果を返す。

    Node.js プロセスを 1回だけ起動し、WASM を使い回して終局まで打ち切る。
    （旧実装は手数分だけプロセスを起動していたため低速だった）

    Parameters
    ----------
    board  : 8×8 の二次元リスト (1=黒, -1=白, 0=空)
    player : 手番 (1=黒, -1=白)
    level  : 探索レベル（21 = 全読み推奨）

    Returns
    -------
    dict:
        score      : 最終石差（黒視点: 正=黒勝, 負=白勝）
        black      : 最終黒石数（日本ルール補正後）
        white      : 最終白石数（日本ルール補正後）
        moves      : 最善手順リスト [{'col':str, 'row':int, 'player':int}, ...]
        final_board: 打ち切り後の最終盤面
    """
    return call_ai(
        {'type': 'solve_endgame', 'board': board, 'player': player, 'level': level},
        timeout=120,
    )


# ===== サンプル終盤局面（空きマス 15 個）=====
#
#   a b c d e f g h
# 1 ● ● ● ● ● ● ● ●
# 2 ● ○ ○ ○ ○ ○ ○ ●
# 3 ● ○ ● ● ● ● ○ ●
# 4 ● ○ ● ○ ○ ● ○ ●
# 5 ● ○ ● ● ○ ● ○ .   ← h5 が空き
# 6 ● ○ ○ ● ○ . . .   ← f6 g6 h6 が空き
# 7 ● ● ● ● . . . .   ← e7 f7 g7 h7 が空き
# 8 ● . . . . . . .   ← b8〜h8 が空き
#
# 合計: 1+3+4+7 = 15 空き  黒31手・白18手

endgame_board = [
    [ 0,  1,  1,  1,  1,  1,  1,  1],
    [ 1,  0, -1, -1, -1, -1, -1,  1],
    [ 1, -1,  1,  1,  1,  1, -1,  1],
    [ 1, -1,  1, -1, -1,  1, -1,  1],
    [ 1, -1,  1,  1, -1,  1, -1,  0],  # h5=空き
    [ 1, -1, -1,  1, -1,  0,  0,  0],  # f6,g6,h6=空き
    [ 1,  1,  1,  1,  0,  0,  0,  0],  # e7,f7,g7,h7=空き
    [ 1,  0,  0,  0,  0,  0,  0,  0],  # b8〜h8=空き
]
empty_count = sum(v == 0 for row in endgame_board for v in row)

print('=== 終盤局面（黒番）===')
print_board(endgame_board)
print(f'空きマス数: {empty_count}')

print(f'\nレベル{21} で全読み中（1プロセス・WASM使い回し）...')
t0 = time.time()
result = solve_endgame(endgame_board, player=1, level=21)
elapsed = time.time() - t0
print(f'計算時間: {elapsed:.2f}秒')

print('\n=== 最善手順 ===')
for i, m in enumerate(result['moves']):
    side = '黒' if m['player'] == 1 else '白'
    print(f'  手{i+1:>2}: {side} → {m["col"]}{m["row"]}')

print('\n=== 最終盤面 ===')
print_board(result['final_board'])

score = result['score']
winner = '黒' if score > 0 else '白' if score < 0 else '引き分け'
print(f'\n最終結果: 黒 {result["black"]} - 白 {result["white"]}')
print(f'勝者: {winner}  (石差: {score:+d})')

=== 終盤局面（黒番）===
  a b c d e f g h
1 . ● ● ● ● ● ● ● 
2 ● . ○ ○ ○ ○ ○ ● 
3 ● ○ ● ● ● ● ○ ● 
4 ● ○ ● ○ ○ ● ○ ● 
5 ● ○ ● ● ○ ● ○ . 
6 ● ○ ○ ● ○ . . . 
7 ● ● ● ● . . . . 
8 ● . . . . . . . 
   黒: 30  白: 17
空きマス数: 17

レベル21 で全読み中（1プロセス・WASM使い回し）...
計算時間: 2.55秒

=== 最善手順 ===
  手 1: 黒 → b2
  手 2: 白 → a1
  手 3: 黒 → h6
  手 4: 白 → h5
  手 5: 黒 → g6
  手 6: 白 → d8
  手 7: 黒 → e8
  手 8: 白 → f8
  手 9: 黒 → f6
  手10: 白 → e7
  手11: 黒 → f7
  手12: 白 → h7
  手13: 黒 → h8
  手14: 白 → b8
  手15: 黒 → c8
  手16: 白 → g8
  手17: 黒 → g7

=== 最終盤面 ===
  a b c d e f g h
1 ○ ● ● ● ● ● ● ● 
2 ● ○ ● ● ● ● ● ● 
3 ● ○ ○ ● ● ● ● ● 
4 ● ○ ○ ○ ● ● ● ● 
5 ● ○ ● ○ ○ ● ● ● 
6 ● ○ ● ○ ○ ● ● ● 
7 ● ● ● ● ● ● ● ● 
8 ● ● ● ○ ○ ○ ○ ● 
   黒: 47  白: 17

最終結果: 黒 47 - 白 17
勝者: 黒  (石差: +30)


In [43]:
def play_game(level: int = 5, max_moves: int = 60) -> list[list[int]]:
    """
    Egaroucid AI 同士でオセロを対局させ、最終盤面を返す。

    Parameters
    ----------
    level     : 探索深さ（小さいほど速い）
    max_moves : 最大手数
    """
    board  = initial_board()
    player = 1  # 黒先手
    history = []

    for move_num in range(max_moves):
        moves = valid_moves(board, player)

        if not moves:
            # 合法手なし → 相手にパス
            opp_moves = valid_moves(board, -player)
            if not opp_moves:
                print(f'終局（{move_num}手目）')
                break
            side = '黒' if player == 1 else '白'
            print(f'{side} パス')
            player = -player
            continue

        # AI に最善手を問い合わせ
        result = best_move(board, player=player, level=level)
        x, y = result['mx'], result['my']
        col   = 'abcdefgh'[x]
        side  = '黒' if player == 1 else '白'
        print(f'手{move_num+1:>2}: {side} → {col}{y+1}  (評価値: {result["score"]:+d})')

        board  = apply_move(board, x, y, player)
        history.append((x, y, player))
        player = -player

    return board


print('=== AI対局開始 (level=5) ===')
final_board = play_game(level=5)
print('\n=== 最終盤面 ===')
print_board(final_board)

=== AI対局開始 (level=5) ===
手 1: 黒 → f5  (評価値: +0)
手 2: 白 → d6  (評価値: +0)
手 3: 黒 → c4  (評価値: +0)
手 4: 白 → g5  (評価値: +0)
手 5: 黒 → f6  (評価値: +0)
手 6: 白 → f4  (評価値: +0)
手 7: 黒 → f3  (評価値: +0)
手 8: 白 → d3  (評価値: +0)
手 9: 黒 → c3  (評価値: +0)
手10: 白 → g6  (評価値: +0)
手11: 黒 → e3  (評価値: +0)


KeyboardInterrupt: 

---
## 7. 任意の局面を指定して評価する

棋譜を手動で入力し、その局面を評価する例です。

In [ ]:
def board_from_kifu(kifu: str) -> tuple[list[list[int]], int]:
    """
    棋譜文字列から盤面と手番を復元する。

    Parameters
    ----------
    kifu : 棋譜文字列。例: 'd3c3c4d6c5f6e6f5f4f3e3e7c6g5'
           小文字 'a'〜'h' + 数字 '1'〜'8' を交互に並べる。

    Returns
    -------
    (board, player) : 最終局面と次の手番
    """
    board  = initial_board()
    player = 1  # 黒先手

    i = 0
    while i + 1 < len(kifu):
        col_ch = kifu[i]
        row_ch = kifu[i + 1]
        i += 2

        x = 'abcdefgh'.index(col_ch)
        y = int(row_ch) - 1

        flips = get_flips(board, x, y, player)
        if not flips:
            # 合法手でない場合はパスとみなして手を再試行
            player = -player
            flips  = get_flips(board, x, y, player)

        board  = apply_move(board, x, y, player)
        player = -player

    return board, player


# --- 実行例 ---
# 有名な「花月定石」の一部を入力
kifu = 'd3c4d2c3e2f2'
board, next_player = board_from_kifu(kifu)

print(f'棋譜: {kifu}')
print(f'次の手番: {"黒" if next_player == 1 else "白"}')
print_board(board)

# この局面の最善手と評価値を問い合わせ
result = best_move(board, player=next_player, level=10)
print(f'\nEgaroucid の最善手: {result["col"]}{result["row"]}  評価値: {result["score"]:+d}')

---
## 8. クリーンアップ

In [ ]:
# 一時ブリッジファイルを削除
if BRIDGE_FILE.exists():
    BRIDGE_FILE.unlink()
    print(f'削除: {BRIDGE_FILE}')